## Preliminary Data Analysis - Milestone 1

In [2]:
import numpy as np
import pandas as pd
import os

In [3]:
# load datasets

# all datasets are keyed by ISO alpha-3 codes

data_dir = "../data"

iso_codes = pd.read_csv(f"{data_dir}/iso/countries.csv", index_col="alpha-3")

worldbank_header_size = 4

gdp = pd.read_csv(f"{data_dir}/worldbank/gdp-current-usd-2026.csv", skiprows=worldbank_header_size, index_col="Country Code")
gdp_ppp = pd.read_csv(f"{data_dir}/worldbank/gdp-ppp-international-usd-2021.csv", skiprows=worldbank_header_size, index_col="Country Code")
gdp_pc = pd.read_csv(f"{data_dir}/worldbank/gdp-capita-current-usd-2026.csv", skiprows=worldbank_header_size, index_col="Country Code")
gdp_pc_ppp = pd.read_csv(f"{data_dir}/worldbank/gdp-capita-ppp-international-usd-2021.csv", skiprows=worldbank_header_size, index_col="Country Code")

# drop 2025 from all datasets since the data is missing
gdp = gdp.drop(columns=["2025"])
gdp_ppp = gdp_ppp.drop(columns=["2025"])
gdp_pc = gdp_pc.drop(columns=["2025"])
gdp_pc_ppp = gdp_pc_ppp.drop(columns=["2025"])

In [4]:
# top 25 countries by GDP in 2024
top_gdp_2024 = gdp[gdp.index.isin(iso_codes.index)].nlargest(25, "2024")

def get_oldest_year_with_data(df):
    nonempty_cols = df.columns[df.notna().all()]
    numeric_cols = nonempty_cols[nonempty_cols.str.isdigit()]
    if len(numeric_cols) == 0:
        return None
    return numeric_cols.min()

# oldest year from which we have contiguous data for the 25
oldest_year = get_oldest_year_with_data(top_gdp_2024)

assert(oldest_year == "1990")
assert(pd.isna(gdp.loc["POL", str(int(oldest_year) - 1)])) # pol is available from 1990
assert(pd.isna(gdp.loc["RUS", str(int(oldest_year) - 3)])) # rus is available from 1988

In [5]:
# for top countries, what are the oldest years for which we have data for all 4 datasets?

datasets = {"gdp": gdp, "gdp_ppp": gdp_ppp, "gdp_pc": gdp_pc, "gdp_pc_ppp": gdp_pc_ppp}

# map to oldest year after filtering to the top 25
oldest_years = {name: get_oldest_year_with_data(ds.loc[top_gdp_2024.index]) for name, ds in datasets.items()}

# all data happens to be available from 1990!
assert(all(year == oldest_year for year in oldest_years.values()))

oldest_years

{'gdp': '1990', 'gdp_ppp': '1990', 'gdp_pc': '1990', 'gdp_pc_ppp': '1990'}

## NASDAQ Data

In [6]:
# import symbol metadata

symbols = pd.read_csv(f"{data_dir}/nasdaq/symbols-valid-meta.csv", index_col="Symbol")

etf_path = f"{data_dir}/nasdaq/etf"

# get every csv from etf_path and load it into a dict of dataframes keyed by symbol
etf_dfs = {}
for filename in os.listdir(etf_path):
    if filename.endswith(".csv"):
        symbol = filename[:-4] # basename 
        df = pd.read_csv(f"{etf_path}/{filename}", index_col="Date", parse_dates=True)
        etf_dfs[symbol] = df

In [7]:
# starting dates for each symbol
start_dates = pd.DataFrame({    
    "Symbol": list(etf_dfs.keys()),
    "Start Date": [df.index.min() for df in etf_dfs.values()],
    "Name" : [symbols.loc[symbol, "Security Name"] if symbol in symbols.index else None for symbol in etf_dfs.keys()]
}).set_index("Symbol")

# stupidly extract the relevant country
# by taking the word after MSCI in the name, if it exists
def extract_country(name):
    if pd.isna(name):
        return None
    parts = name.split()
    if "MSCI" in parts:
        idx = parts.index("MSCI")
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return None

start_dates["Country"] = start_dates["Name"].apply(extract_country)

start_dates.sort_values("Start Date").head(10)

,Start Date,Name,Country
Symbol,,,
EWD,1996-03-18,iShares Inc iShares MSCI Sweden ETF,Sweden
EWN,1996-03-18,iShares MSCI Netherlands Index Fund,Netherlands
EWO,1996-03-18,iShares Inc iShares MSCI Austria ETF,Austria
EWM,1996-03-18,iShares MSCI Malaysia Index Fund,Malaysia
EWW,1996-03-18,iShares Inc iShares MSCI Mexico ETF,Mexico
EWK,1996-03-18,iShares Inc iShares MSCI Belgium ETF,Belgium
EWJ,1996-03-18,iShares MSCI Japan Index Fund,Japan
EWS,1996-03-18,iShares Inc iShares MSCI Singapore ETF,Singapore
EWA,1996-03-18,iShares MSCI Australia Index Fund,Australia


## Stock Listing Data (General)

Checking the larger data set (AMEX/NYSE/NASDAQ) and combining it with stock listing data from NASDAQ.

In [32]:
# load the symbol data from NASDAQ

nasdaq_only_symbols = pd.read_csv(f"{data_dir}/stock/nasdaq-listed.csv", index_col="Symbol", sep="|") 
other_listed_symbols = pd.read_csv(f"{data_dir}/stock/other-listed.csv", index_col="NASDAQ Symbol", sep="|")

# clean the metadata from both datasets
nan_symbols_nasdaq = nasdaq_only_symbols[nasdaq_only_symbols.index.str.startswith("File Creation Time")]
assert(len(nan_symbols_nasdaq) == 1)

nan_symbols_other = other_listed_symbols[other_listed_symbols.index.isna()]
assert(len(nan_symbols_other) == 1)
assert(nan_symbols_other["ACT Symbol"].apply(lambda s: s.startswith("File Creation Time")).all())

# drop the rows with the nan symbols
nasdaq_only_symbols = nasdaq_only_symbols.drop(index=nan_symbols_nasdaq.index)
other_listed_symbols = other_listed_symbols.drop(index=nan_symbols_other.index)

# are there any symbols that are in both datasets?
overlap_symbols = set(nasdaq_only_symbols.index).intersection(set(other_listed_symbols.index))
assert(len(overlap_symbols) == 0)

In [ ]:
# preprocess and merge the listings

nasdaq_only_symbols["Exchange"] = "N"

# we only care about the following columns
## NASDAQ: Symbol [index], Security Name, Exchange, ETF
## Other: NASDAQ Symbol [index], Security Name, Exchange, ETF

# TODO/FIXME: how is market category defined? we only have this data 
# for NASDAQ, but maybe it's less relevant for ETFs regardless

nasdaq_chosen = nasdaq_only_symbols[["Security Name", "Exchange", "ETF"]]
other_chosen = other_listed_symbols[["Security Name", "Exchange", "ETF"]]

listings = pd.concat([nasdaq_chosen, other_chosen], axis=0)

# check for no data being na in any column
assert(not listings.isna().any().any())

# turn ETF N/Y into boolean
listings["ETF"] = listings["ETF"].map({"Y": True, "N": False})

print(f"Unique exchanges: {listings['Exchange'].unique()}")

listings.head()

Unique exchanges: <StringArray>
['N', 'P', 'Z', 'A', 'M', 'V']
Length: 6, dtype: str


,Security Name,Exchange,ETF
AACB,Artius II Acquisition Inc. - Class A Ordinary ...,N,False
AACBR,Artius II Acquisition Inc. - Rights,N,False
AACBU,Artius II Acquisition Inc. - Units,N,False
AACG,ATA Creativity Global - American Depositary Sh...,N,False
AACIU,Armada Acquisition Corp. III - Units,N,False
